# Week 5, Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

Moves the Day 2 LangChain agent to LangGraph: an explicit graph of nodes and
edges instead of a single `AgentExecutor` loop, with real branching, a
self-correction cycle, a human-in-the-loop interrupt, and persistence.

**Scenario (kept consistent with Day 2):** a budget-recommendation assistant
that retrieves Laptop A / Laptop B pricing, drafts a recommendation, critiques
its own draft (looping back to improve it if incomplete), pauses for human
approval before "sending" the recommendation email (the risky, hard-to-undo
action), then formats the final answer.


# Task 1 — Graph Concepts & State Design

Before building the workflow, let's understand the main components of LangGraph. Unlike `AgentExecutor`, where the reasoning loop is mostly hidden, LangGraph lets us explicitly define each step of the workflow and control how execution moves between them.

## Core Building Blocks

- **StateGraph** – The main graph builder in LangGraph. It defines the overall workflow by specifying the shared state, adding nodes, connecting them with edges, and finally compiling everything into a runnable graph.

- **Node** – A Python function that performs a single task. Each node receives the current shared state, processes it, and returns only the fields it wants to update. LangGraph automatically merges these updates into the shared state.

- **Edge** – A fixed transition between two nodes. After one node finishes executing, the graph always moves to the connected node.

- **Conditional Edge** – A transition whose destination depends on the current state. Instead of always following the same path, a routing function decides which node should execute next. This enables branching, retries, and self-correcting loops.

- **State** – A shared object (usually a `TypedDict` or `Pydantic` model) that stores all information required throughout the workflow. Every node can read from it and update it, making it the single source of truth for the graph.

## State Schema for This Workflow

This notebook implements a **Budget Recommendation Assistant**. The workflow retrieves product information, generates a recommendation, critiques its own output, optionally loops back for improvement, pauses for human approval before sending an email, and finally formats the result.

```python
class ResearchState(TypedDict):
    question: str
    plan: str
    laptop_a: dict
    laptop_b: dict
    draft: str
    critique_feedback: str
    quality_score: float
    retries: int
    max_retries: int
    human_decision: Optional[str]   # "approve" | "reject" | None
    email_status: str
    final_answer: str
    log: List[str]                  # our own debugging trail, not sent to any LLM
```

### The graph, before writing any code

```
 (START)
    |
   plan
    |
 retrieve
    |
 generate <----------------.
    |                       |
 critique --- incomplete ---'   (conditional edge; bounded by max_retries)
    |
  (complete)
    |
 send_email   <-- INTERRUPT: pauses here for human approval (Task 4)
    |
  format
    |
  (END)
```

This graph starts with planning and information retrieval, generates a recommendation, critiques its quality, and repeats the generation step if necessary. Once the recommendation is considered acceptable, the workflow pauses for human approval before performing the simulated email action and formatting the final response.

A rendered version of this exact graph (auto-generated from the *compiled*
LangGraph object, not hand-drawn) appears at the end of Task 3 below, once
the graph actually exists.

# Task 2 — Build a Linear Graph

Now that the workflow has been designed, the next step is to implement a simple **linear LangGraph**.

This graph follows a fixed sequence of execution:

```
START → Plan → Retrieve → Generate → Format → END
```

Unlike the workflow we'll build in Task 3, this version has **no branching or loops**. Every node executes exactly once before moving to the next node.

The retrieval step reuses the product lookup logic from Day 2. However, instead of relying on an LLM to decide when to call a tool, the graph explicitly controls the execution order by invoking the appropriate node at the correct stage.

Finally, the graph is compiled and executed on a sample input while printing the shared state after each node. This makes it easy to verify that each node correctly updates the workflow state.

In [16]:
"""
Shared state schema used throughout the LangGraph workflow.

This TypedDict defines all the information that can be stored and updated
as the graph executes. Each node reads from this shared state and returns
partial updates, which LangGraph automatically merges into the existing state.
"""

from typing import TypedDict, Optional, List


class ResearchState(TypedDict):
    # ---- input ----
    question: str

    # ---- plan node output ----
    plan: str

    # ---- retrieve node output ----
    laptop_a: dict
    laptop_b: dict

    # ---- generate / critique loop ----
    draft: str
    critique_feedback: str
    quality_score: float
    retries: int
    max_retries: int

    # ---- human-in-the-loop ----
    human_decision: Optional[str]  # "approve" | "reject" | None (pending)
    email_status: str

    # ---- output ----
    final_answer: str

    # ---- our own debugging log (working-memory style, not sent to any LLM) ----
    log: List[str]


In [48]:
"""
Tools reused from Day 2, reimplemented as plain functions (a LangGraph node
calls whatever Python it needs directly -- there's no requirement to route
through an LLM's tool-calling interface when the *graph structure itself*
already decides what runs when).
"""

import json
import os

PRODUCTS_PATH = os.path.join(os.getcwd(), "products.json")
with open(PRODUCTS_PATH) as _f:
    _PRODUCTS = json.load(_f)


def get_product_price(product_name: str) -> dict:
    """Look up a product's price/brand/category from the local catalog."""
    key = product_name.strip().lower().replace(" ", "_")
    record = _PRODUCTS.get(key)
    if not record:
        raise ValueError(f"'{product_name}' not found in catalog")
    return {"product": key, **record}


In [ ]:
"""
Node functions. Each takes the current State dict and returns a partial dict
of updates -- LangGraph merges these into the shared State automatically.
"""


def plan_node(state):
    log = list(state.get("log", []))
    plan = ("1) retrieve prices for Laptop A and Laptop B  "
            "2) draft a recommendation  3) critique it for completeness  "
            "4) get human approval before emailing the client  5) format the final answer")
    log.append(f"[plan] {plan}")
    return {"plan": plan, "log": log}


def retrieve_node(state):
    log = list(state.get("log", []))
    a = get_product_price("laptop_a")
    b = get_product_price("laptop_b")
    log.append(f"[retrieve] laptop_a={a}")
    log.append(f"[retrieve] laptop_b={b}")
    return {"laptop_a": a, "laptop_b": b, "log": log}


def generate_node(state):
    log = list(state.get("log", []))

    retries = state.get("retries", 0)

    laptop_a = state["laptop_a"]
    laptop_b = state["laptop_b"]

    # Find the cheaper laptop
    if laptop_a["price"] <= laptop_b["price"]:
        recommended = laptop_a
        other = laptop_b
    else:
        recommended = laptop_b
        other = laptop_a

    price_difference = abs(laptop_a["price"] - laptop_b["price"])

    # ---------- FIRST PASS ----------
    # Purposely generate an incomplete draft.
    if retries == 0:

        draft = (
            f"I recommend {recommended['product']}."
        )

    # ---------- SECOND PASS ----------
    # After critique requests improvements, generate a complete answer.
    else:

        draft = (
            f"I recommend {recommended['product']} "
            f"({recommended['brand']}, ${recommended['price']}) "
            f"over {other['product']} "
            f"({other['brand']}, ${other['price']}) "
            f"because it is ${price_difference} cheaper while still "
            f"meeting the client's budget requirements."
        )

    log.append(f"[generate] retry={retries} draft={draft}")

    return {
        "draft": draft,
        "log": log,
    }


def critique_node(state):
    log = list(state.get("log", []))
    draft = state["draft"]
    retries = state.get("retries", 0)

    mentions_both_prices = str(state["laptop_a"]["price"]) in draft and str(state["laptop_b"]["price"]) in draft
    mentions_reason = "since" in draft.lower() or "because" in draft.lower()
    score = (0.5 if mentions_both_prices else 0.0) + (0.5 if mentions_reason else 0.0)

    missing = []
    if not mentions_both_prices:
        missing.append("both product prices")
    if not mentions_reason:
        missing.append("an explicit reason")
    feedback = "complete" if score >= 1.0 else "missing " + ", ".join(missing)

    log.append(f"[critique] pass={retries + 1} score={score} feedback='{feedback}'")
    return {"quality_score": score, "critique_feedback": feedback, "retries": retries + 1, "log": log}


def route_after_critique(state):
    """Conditional edge: loop back to 'generate' until quality passes,
    bounded by max_retries so a persistently-bad draft can't loop forever."""
    if state["quality_score"] >= 1.0:
        return "send_email"
    if state["retries"] >= state["max_retries"]:
        return "send_email"  # guardrail: give up looping, proceed with best effort
    return "generate"


def send_email_node(state):
    log = list(state.get("log", []))
    decision = state.get("human_decision")

    if decision == "approve":
        status = f"Email SENT to client: \"{state['draft']}\""
    elif decision == "reject":
        status = "Email NOT sent -- human reviewer rejected the recommendation."
    else:
        status = "Email NOT sent -- no human decision was recorded."

    log.append(f"[send_email] human_decision={decision!r} -> {status}")
    return {"email_status": status, "log": log}

def format_node(state): # for task 3
    log = list(state.get("log", []))

    final = (
        f"{state['draft']}\n\n"
        f"[Email status: {state['email_status']}]"
    )

    log.append("[format] final answer assembled")

    return {
        "final_answer": final,
        "log": log,
    }

def format_node_linear(state): # for task 2
    log = list(state.get("log", []))

    final = (
        "=== Budget Recommendation ===\n\n"
        f"{state['draft']}"
    )

    log.append("[format] Final answer assembled.")

    return {
        "final_answer": final,
        "log": log,
    }


In [49]:
"""
Task 2 -- a simple linear 4-node graph: plan -> retrieve -> generate -> format.
No conditional edges or loops yet -- those come in Task 3.
"""

from langgraph.graph import StateGraph, START, END

# For this linear-only version, "format" just wraps the draft directly --
# there's no email/approval step yet (that arrives in Task 4).
def format_node_linear(state):
    log = list(state.get("log", []))
    final = state["draft"]
    log.append("[format] final answer assembled (linear version, no approval step)")
    return {"final_answer": final, "log": log}


def build_linear_graph():
    graph = StateGraph(ResearchState)
    graph.add_node("plan", plan_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("generate", generate_node)
    graph.add_node("format", format_node_linear)

    graph.add_edge(START, "plan")
    graph.add_edge("plan", "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", "format")
    graph.add_edge("format", END)

    return graph.compile()




> **Note:** Each node returns only the fields it wants to update. LangGraph automatically merges those updates into the existing shared state, so previously stored information (such as the user's question) remains available to all subsequent nodes.

In [50]:
app = build_linear_graph()

initial_state = {
    "question": "Which laptop should I recommend to a budget-conscious client?",
    "retries": 0,
    "max_retries": 3,
    "log": [],
}

print("=" * 70)
print("Streaming graph execution")
print("=" * 70)

for i, state in enumerate(app.stream(initial_state, stream_mode="values"), start=1):

    print(f"\nStep {i}")

    visible = {k: v for k, v in state.items() if k != "log"}

    for key, value in visible.items():
        print(f"{key}: {value}")

    if state.get("log"):
        print(f"Latest log: {state['log'][-1]}")

print("\n=== FINAL STATE ===")
final = app.invoke(initial_state)
print(final["final_answer"])

Streaming graph execution

Step 1
question: Which laptop should I recommend to a budget-conscious client?
retries: 0
max_retries: 3

Step 2
question: Which laptop should I recommend to a budget-conscious client?
plan: 1) retrieve prices for Laptop A and Laptop B  2) draft a recommendation  3) critique it for completeness  4) get human approval before emailing the client  5) format the final answer
retries: 0
max_retries: 3
Latest log: [plan] 1) retrieve prices for Laptop A and Laptop B  2) draft a recommendation  3) critique it for completeness  4) get human approval before emailing the client  5) format the final answer

Step 3
question: Which laptop should I recommend to a budget-conscious client?
plan: 1) retrieve prices for Laptop A and Laptop B  2) draft a recommendation  3) critique it for completeness  4) get human approval before emailing the client  5) format the final answer
laptop_a: {'product': 'laptop_a', 'price': 799, 'brand': 'Dell'}
laptop_b: {'product': 'laptop_b', 'pr

### Observation

The streamed execution confirms that the graph executes each node in the expected order:

**Plan → Retrieve → Generate → Format**

After each node, LangGraph automatically merges the returned updates into the shared state instead of replacing it. As a result, information produced by earlier nodes remains available to all later nodes, demonstrating how the shared state evolves throughout the workflow.

## Task 3 — Add Conditional Edges & Cycles

The linear workflow from Task 2 is now extended with a **critique** node and a **conditional edge**.

Instead of always proceeding to the next step, the graph evaluates the generated recommendation. If the draft is incomplete, the graph loops back to the `generate` node for another attempt. Otherwise, it continues to the next stage.

To prevent infinite loops, the shared state maintains both a `retries` counter and a `max_retries` limit. Each pass through the loop is recorded in the execution log, making the graph's behavior easy to observe and debug.

### Updating the `generate` node

For the linear graph in Task 2, the `generate` node always produced a complete recommendation.

To demonstrate LangGraph's self-correction capabilities, we now modify the node so that the **first draft is intentionally incomplete**. The `critique` node will detect the missing information and, if necessary, route execution back to `generate` to produce an improved draft. This allows us to demonstrate a conditional loop while preventing infinite cycles using `max_retries`.


In [51]:
def generate_node_task3(state):
    log = list(state.get("log", []))

    retries = state.get("retries", 0)

    laptop_a = state["laptop_a"]
    laptop_b = state["laptop_b"]

    if retries == 0:
        # Intentionally incomplete first draft
        draft = f"I recommend {laptop_a['product']}."
    else:
        if laptop_a["price"] <= laptop_b["price"]:
            recommended = laptop_a
            other = laptop_b
        else:
            recommended = laptop_b
            other = laptop_a

        price_difference = abs(laptop_a["price"] - laptop_b["price"])

        draft = (
            f"I recommend {recommended['product']} ({recommended['brand']}, "
            f"${recommended['price']}) over {other['product']} "
            f"({other['brand']}, ${other['price']}) because it is "
            f"${price_difference} cheaper while still meeting the client's "
            f"budget requirements."
        )

    log.append(f"[generate] retry={retries} -> {draft}")

    return {
        "draft": draft,
        "log": log,
    }

In [52]:
"""
Task 3 -- add a conditional edge: 'critique' routes back to 'generate' if the
draft is incomplete, or forward if it passes -- a self-correction loop, bounded
by max_retries so it can't cycle forever.
"""

from langgraph.graph import StateGraph, START, END

def build_looping_graph():
    graph = StateGraph(ResearchState)
    graph.add_node("plan", plan_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("generate", generate_node_task3)
    graph.add_node("critique", critique_node)
    graph.add_node("send_email", send_email_node)
    graph.add_node("format", format_node)

    graph.add_edge(START, "plan")
    graph.add_edge("plan", "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", "critique")
    graph.add_conditional_edges(
        "critique", route_after_critique,
        {"generate": "generate", "send_email": "send_email"},
    )
    graph.add_edge("send_email", "format")
    graph.add_edge("format", END)

    return graph




In [53]:
app = build_looping_graph().compile()

initial_state = {
    "question": "Which laptop should I recommend to a budget-conscious client?",
    "retries": 0,
    "max_retries": 3,
    "human_decision": "approve",  # no interrupt yet in this version -- added next
    "log": [],
}

final = app.invoke(initial_state)

print("=" * 70)
print("Execution Log")
print("=" * 70)

for i, line in enumerate(final["log"], start=1):
    print(f"{i}. {line}")

print("\n" + "=" * 70)
print("Final Answer")
print("=" * 70)
print(final["final_answer"])

Execution Log
1. [plan] 1) retrieve prices for Laptop A and Laptop B  2) draft a recommendation  3) critique it for completeness  4) get human approval before emailing the client  5) format the final answer
2. [retrieve] laptop_a={'product': 'laptop_a', 'price': 799, 'brand': 'Dell'}
3. [retrieve] laptop_b={'product': 'laptop_b', 'price': 999, 'brand': 'HP'}
4. [generate] retry=0 -> I recommend laptop_a.
5. [critique] pass=1 score=0.0 feedback='missing both product prices, an explicit reason'
6. [generate] retry=1 -> I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.
7. [critique] pass=2 score=1.0 feedback='complete'
8. [send_email] human_decision='approve' -> Email SENT to client: "I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements."
9. [format] final answer assembled

Final Answer
I recommend laptop_a (Dell, $

### Why is this easier in LangGraph than in `AgentExecutor`?

In a traditional `AgentExecutor`, the reasoning loop is controlled implicitly by the LLM, making it difficult to enforce that a specific step should repeat under certain conditions. Developers often have to rely on prompt instructions and hope the model follows them consistently. In LangGraph, conditional edges make this behavior explicit by allowing the workflow to route directly from one node to another based on the current state, making self-correcting workflows much easier to build and maintain.

### The rendered graph (auto-generated from the compiled object above)


In [54]:
print(build_looping_graph().compile().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	retrieve(retrieve)
	generate(generate)
	critique(critique)
	send_email(send_email)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	critique -.-> generate;
	critique -.-> send_email;
	generate --> critique;
	plan --> retrieve;
	retrieve --> generate;
	send_email --> format;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



<h3>Auto-generated Workflow Diagram</h3>

<p>
The following diagram represents the final LangGraph workflow. The Mermaid
definition was generated automatically from the compiled graph using
<code>draw_mermaid()</code>, then rendered as a PNG for readability.
</p>

<img src="mermaid_diagram.png" width="300" height="700">

## Task 4 — Human-in-the-Loop & Interrupts

This workflow introduces a **human approval checkpoint** before the
`send_email` node. Using LangGraph's `interrupt_before`, execution pauses
immediately before the risky action, allowing a human to review the generated
recommendation.

The graph is then resumed after simulating both possible outcomes:

- **Approve** → the email is sent.
- **Reject** → the email is not sent.

This demonstrates how LangGraph supports human oversight while preserving the
graph's execution state through a checkpointer (`MemorySaver`).

In [55]:
"""
Task 4 -- Human-in-the-Loop

Pause execution before the risky "send_email" node using
interrupt_before. A human reviews the draft, then the graph
resumes after either approving or rejecting it.
"""

from langgraph.checkpoint.memory import MemorySaver

# Reuse the graph from Task 3
checkpointer = MemorySaver()

app = build_looping_graph().compile(
    checkpointer=checkpointer,
    interrupt_before=["send_email"]
)

In [56]:
def run_human_review(decision):
    """
    Runs the graph until the interrupt point, simulates a human
    approval/rejection, then resumes execution.
    """

    config = {
        "configurable": {
            "thread_id": f"{decision}-demo"
        }
    }

    initial_state = {
        "question": "Which laptop should I recommend to a budget-conscious client?",
        "retries": 0,
        "max_retries": 3,
        "human_decision": None,
        "log": [],
    }

    print("=" * 70)
    print("Graph Execution (Paused for Human Review)")
    print("=" * 70)

    # Execute until interrupt_before("send_email")
    result = app.invoke(initial_state, config=config)

    snapshot = app.get_state(config)

    next_node = snapshot.next[0] if snapshot.next else "None"
    print(f"Execution paused before node: {next_node}")
    print(f"\nDraft awaiting human review:\n")
    print(result["draft"])

    print("=" * 70)
    print(f"Simulated Human Decision: {decision.upper()}")
    print("=" * 70)

    # Simulate human decision
    app.update_state(
        config,
        {"human_decision": decision}
    )

    # Resume execution
    final = app.invoke(None, config=config)

    print("\nFinal Answer\n")
    print(final["final_answer"])

    return final

In [57]:
# Demonstration: Human approves the recommendation

approve_result = run_human_review("approve")

Graph Execution (Paused for Human Review)
Execution paused before node: send_email

Draft awaiting human review:

I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.
Simulated Human Decision: APPROVE

Final Answer

I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.

[Email status: Email SENT to client: "I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements."]


In [58]:
print("\n" + "=" * 80 + "\n")

# Demonstration: Human rejects the recommendation

reject_result = run_human_review("reject")



Graph Execution (Paused for Human Review)
Execution paused before node: send_email

Draft awaiting human review:

I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.
Simulated Human Decision: REJECT

Final Answer

I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.

[Email status: Email NOT sent -- human reviewer rejected the recommendation.]


### Discussion

Human-in-the-loop is appropriate whenever an agent performs actions that are
costly, irreversible, or have real-world consequences, such as sending emails,
processing payments, deleting data, or making medical or legal decisions.
Human review provides an additional safety layer before these actions occur.

Full autonomy is suitable for low-risk and easily reversible tasks, such as
retrieving information, summarizing text, performing calculations, or drafting
content for later review. These tasks have minimal consequences if the agent
makes a mistake.

## Task 5 — Persistence & Debugging

This task demonstrates two important LangGraph capabilities:

- **Persistence:** A `MemorySaver` checkpointer stores the graph's state using a unique `thread_id`. Even after execution pauses, the workflow can later resume from exactly the same point.
- **Time-travel debugging:** LangGraph records state checkpoints throughout execution. These snapshots allow developers to inspect previous states, replay execution from an earlier point, and debug complex workflows more easily.

In [63]:
"""
Task 5 -- Persistence (resuming a paused conversation) and time-travel
debugging (replaying an earlier checkpoint).
"""

def demonstrate_persistence():

    config = {
        "configurable": {
            "thread_id": "persistent-session"
        }
    }

    print("=" * 70)
    print("Session 1: Run until interruption")
    print("=" * 70)

    app.invoke(
        {
            "question": "Which laptop should I recommend to a budget-conscious client?",
            "retries": 0,
            "max_retries": 3,
            "human_decision": None,
            "log": [],
        },
        config=config,
    )

    print("\nExecution paused and state saved.\n")

    print("=" * 70)
    print("Session 2: Recover saved state")
    print("=" * 70)

    snapshot = app.get_state(config)

    next_node = snapshot.next[0] if snapshot.next else "END"

    print(f"Recovered next node: {next_node}")
    print(f"Recovered draft:\n{snapshot.values['draft']}")

    print("\nResuming execution...\n")

    app.update_state(
        config,
        {"human_decision": "approve"}
    )

    final = app.invoke(None, config=config)

    print(final["email_status"])

    return config


def demonstrate_time_travel(config):

    print("=" * 70)
    print("Checkpoint History")
    print("=" * 70)

    history = list(app.get_state_history(config))

    for i, snapshot in enumerate(reversed(history), start=1):

        next_node = snapshot.next[0] if snapshot.next else "END"

        retries = snapshot.values.get("retries", 0)

        print(
            f"Checkpoint {i}: "
            f"Next = {next_node:<12} "
            f"Retries = {retries}"
        )

    print("\nReplaying an earlier checkpoint...\n")

    target = None

    for snapshot in history:

        if (
            snapshot.values.get("retries") == 0
            and snapshot.values.get("draft")
        ):
            target = snapshot
            break

    if target is None:
        print("No replayable checkpoint found.")
        return

    replay = app.invoke(None, config=target.config)

    print("Draft after replay:\n")
    print(replay["draft"])




In [64]:
cfg = demonstrate_persistence()

Session 1: Run until interruption

Execution paused and state saved.

Session 2: Recover saved state
Recovered next node: send_email
Recovered draft:
I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements.

Resuming execution...

Email SENT to client: "I recommend laptop_a (Dell, $799) over laptop_b (HP, $999) because it is $200 cheaper while still meeting the client's budget requirements."


In [65]:
demonstrate_time_travel(cfg)

Checkpoint History
Checkpoint 1: Next = __start__    Retries = 0
Checkpoint 2: Next = plan         Retries = 0
Checkpoint 3: Next = retrieve     Retries = 0
Checkpoint 4: Next = generate     Retries = 0
Checkpoint 5: Next = critique     Retries = 0
Checkpoint 6: Next = generate     Retries = 1
Checkpoint 7: Next = critique     Retries = 1
Checkpoint 8: Next = send_email   Retries = 2
Checkpoint 9: Next = send_email   Retries = 2
Checkpoint 10: Next = format       Retries = 2
Checkpoint 11: Next = END          Retries = 2
Checkpoint 12: Next = critique     Retries = 0
Checkpoint 13: Next = generate     Retries = 1
Checkpoint 14: Next = critique     Retries = 1
Checkpoint 15: Next = send_email   Retries = 2
Checkpoint 16: Next = __start__    Retries = 2
Checkpoint 17: Next = plan         Retries = 0
Checkpoint 18: Next = retrieve     Retries = 0
Checkpoint 19: Next = generate     Retries = 0
Checkpoint 20: Next = critique     Retries = 0
Checkpoint 21: Next = generate     Retries = 1
Che

### LangChain AgentExecutor vs. LangGraph

`AgentExecutor` is ideal for relatively simple agent workflows where the model repeatedly reasons, calls tools when necessary, and produces a final answer. It is easy to set up and works well for straightforward conversational agents or question-answering systems.

LangGraph is better suited for complex workflows that require explicit control over execution, such as conditional branching, self-correction loops, human approval steps, persistent state, or the ability to pause and resume execution. When building production-grade AI systems with multiple stages and long-running workflows, LangGraph provides significantly greater flexibility and control.